# Day 3 — ILT 1: API Ingestion Mechanics + Intro to GraphDB / Cypher Basics
**Time:** 9:30 AM – 11:00 AM
**What comes next:** Hands-on — Connect to Public API + Neo4j, explore and land samples (11:00 AM – 1:00 PM)

---
### Side-exploration, not part of the GlobalMart build
GlobalMart's real pipeline has exactly two sources: **Postgres (Supabase) via Lakeflow Connect CDC** for `orders`/`order_items`, and **ADLS file drops via Autoloader** for `products`/`customers`/`address`/`payments`. A REST API and a graph database are not among them. Today's session steps outside that pipeline on purpose — these are patterns you will meet on *other* projects, and GlobalMart data is just a convenient, familiar example to practice on. Nothing built today feeds Bronze/Silver/Gold or `fact_sales`.

### What we cover today
1. What is a REST API and how do Data Engineers use it
2. How to call an API from Databricks and land JSON in a sandbox table
3. What is a Graph Database and why it exists
4. Cypher query language basics — nodes, relationships, MATCH
5. Free tools: frankfurter.app (API) + Neo4j AuraDB Free (GraphDB)

> **Instructor note:** 90 minutes. ~35 min on API ingestion with live code, ~35 min on GraphDB theory + Cypher, ~20 min wrap-up + Q&A. These are "demo-once" sources — show the concept clearly, students won't build on these every day.

## Section 1 — What is a REST API?

An **API (Application Programming Interface)** is a way for one system to talk to another over the internet.

**REST API** = a specific style of API that uses HTTP (the same protocol as websites).

### How it works — simple version

```
Your code                         API Server
    |                                  |
    |--- HTTP GET /latest ------------>|
    |                                  |
    |<-- JSON response ----------------|
    |
    v
Parse JSON → save to a landing table
```

### Why Data Engineers use APIs

| Use Case | Example API |
|----------|------------|
| Currency exchange rates | frankfurter.app |
| Weather data | OpenWeatherMap |
| Shipping rates | FedEx / UPS APIs |
| Company financials | Alpha Vantage |
| Address validation | Google Maps |

### Why we're using frankfurter.app today
This is a **pattern demo, not a GlobalMart requirement**. GlobalMart's real Gold layer computes revenue straight from the source-currency order amounts — there is no FX conversion step in the actual build. We use **frankfurter.app** (free, no API key) purely because it's a reliable public API to practice the ingestion mechanics on:

```
Step 1: Call API             → get JSON response
Step 2: Parse JSON           → flatten into rows
Step 3: Land in a table      → NOT bronze/ — a separate sandbox/ path
```

Keep this distinction clear: `bronze/customers`, `bronze/payments`, etc. are the real pipeline (Autoloader, Day 3 afternoon). What we build in this session lands in a sandbox location precisely so it can't be mistaken for part of that pipeline.

## Section 2 — Key API Concepts Every DE Must Know

| Concept | What it means | Example |
|---------|---------------|---------|
| **Endpoint** | The URL you call | `https://api.frankfurter.app/latest` |
| **HTTP GET** | Read-only request — fetch data | Most data APIs use GET |
| **HTTP POST** | Send data to the API | Webhooks, form submissions |
| **JSON** | The format APIs return data in | `{"USD": 1.08, "INR": 89.5}` |
| **Status code** | Did it work? | 200 = OK, 404 = not found, 500 = server error |
| **Rate limit** | How many calls allowed per minute | Free APIs: usually 100/day or 60/min |
| **Pagination** | Large results split into pages | `page=1`, `page=2`, `offset=100` |
| **API key** | Password to authenticate | Not needed for frankfurter.app |

### The DE Ingestion Flow for APIs

```
Step 1: Call API              → get JSON response
Step 2: Check status code     → 200 = continue, anything else = raise error
Step 3: Parse JSON            → convert to Spark DataFrame
Step 4: Add audit columns     → ingestion_timestamp, source_url
Step 5: Save to a landing table → Delta format (sandbox/ for this exploration)
```

## Setup — Connect to ADLS

In [ ]:
# ─── ADLS Setup — Unity Catalog External Location, no storage key ─────────────
# Same external location the real pipeline uses (set up Day 2), but we write
# under sandbox/ — deliberately NOT raw-data/ or <your-catalog>.bronze.*. This keeps
# today's exploration physically and logically separate from the real
# GlobalMart medallion pipeline.

EXTERNAL_LOCATION = "abfss://<your-container>@<your-storage-account>.dfs.core.windows.net"
sandbox_path = f"{EXTERNAL_LOCATION}/sandbox/api_graphdb"
print(f"Sandbox path: {sandbox_path}")

## Live Demo — Call the frankfurter.app API

In [ ]:
# ─── Step 1: Call the API and see the raw response ────────────────────────────
import requests
import json

# frankfurter.app gives current exchange rates — free, no API key needed
api_url = "https://api.frankfurter.app/latest"

response = requests.get(api_url)

# Always check status code first — 200 means success
print(f"Status code : {response.status_code}")
print(f"URL called  : {response.url}")
print()

if response.status_code == 200:
    data = response.json()   # parse JSON string into Python dictionary
    print("Raw JSON response:")
    print(json.dumps(data, indent=2))
else:
    print(f"API call failed: {response.status_code}")

In [ ]:
# ─── Step 2: Explore the JSON structure ──────────────────────────────────────
# The JSON has this structure:
# {
#   "amount": 1.0,
#   "base": "EUR",
#   "date": "2024-01-15",
#   "rates": {
#     "AUD": 1.6234,
#     "INR": 89.45,
#     "USD": 1.0876,
#     ... (30+ currencies)
#   }
# }

print(f"Base currency : {data['base']}")
print(f"Rate date     : {data['date']}")
print(f"Total currencies: {len(data['rates'])}")
print()

# Show specific rates relevant to GlobalMart
relevant_currencies = ["USD", "INR", "GBP", "JPY", "AUD"]
print("Exchange rates relevant to GlobalMart:")
for currency in relevant_currencies:
    rate = data["rates"].get(currency, "N/A")
    print(f"  1 {data['base']} = {rate} {currency}")

In [ ]:
# ─── Step 3: Convert JSON to a Spark DataFrame ───────────────────────────────
# APIs return nested JSON — we need to flatten it into rows and columns

from pyspark.sql import Row
from datetime import datetime

# Flatten the nested rates dict — one row per currency
rows = [
    Row(
        base_currency   = data["base"],
        target_currency = currency,
        exchange_rate   = float(rate),
        rate_date       = data["date"],
        ingested_at     = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    )
    for currency, rate in data["rates"].items()
]

fx_df = spark.createDataFrame(rows)

print(f"Exchange rate rows created: {fx_df.count()}")
fx_df.show(10)

In [ ]:
# ─── Step 4: Save to a sandbox landing table ──────────────────────────────────
# Delta format, same mechanics as Bronze — but a sandbox/ path, not bronze/.
# This is a side-exploration output, kept structurally separate from the
# real GlobalMart medallion tables.

fx_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{sandbox_path}/fx_rates")

# Verify
saved = spark.read.format("delta").load(f"{sandbox_path}/fx_rates")
print(f"FX rates saved to sandbox: {saved.count()} rows")
saved.filter("target_currency IN ('USD','INR','GBP')").show()

## Section 3 — Common API Challenges

| Challenge | What happens | How to handle |
|-----------|-------------|---------------|
| **Rate limiting** | API rejects your call after too many requests | Add `time.sleep(1)` between calls, check headers |
| **Pagination** | API only returns 100 rows per call, there are 10,000 | Loop through pages until empty response |
| **Schema change** | API adds/removes fields without warning | Use `inferSchema` carefully, validate response |
| **API downtime** | API returns 500 or 503 | Add retry logic with `try/except` |
| **Auth expiry** | API key or token expires | Refresh token before each run |

### Idempotency for API ingestion

APIs return live data — calling it twice gives different results (rates change daily).  
Always use **`mode("overwrite")`** for API data — you want the latest snapshot, not duplicates.

```
Today's run:     overwrite sandbox table → table has today's rates
Tomorrow's run:  overwrite sandbox table → table has tomorrow's rates (old ones gone)
```

> If you need history, append with a `rate_date` column — one row per currency per day.

## Section 4 — Introduction to Graph Databases

### What is a Graph Database?

A graph database stores data as **nodes** (entities) and **relationships** (connections between entities).

```
Relational DB (tables):          Graph DB (nodes + relationships):

customers table                  (Customer: Raj)
+----+---------+                      |
| id | name    |                   BOUGHT
+----+---------+                      |
| 1  | Raj     |               (Product: iPhone)
| 2  | Priya   |                      |
+----+---------+                 SUPPLIED_BY
                                       |
orders table                    (Supplier: Apple Inc.)
+----+----+------------+
| id | cid| product_id |
+----+----+------------+
```

The graph version makes it **easy to traverse relationships**:  
> *"Which suppliers are connected to customers who returned products?"*  
> In SQL: 4-table join. In graph: one Cypher traversal.

### Illustrative only — modeling GlobalMart-shaped data as a graph

This table is **not a real deliverable** — it's a familiar example to build intuition for how entities you already know (customer, order, product) could be re-modeled as a graph. Nothing here writes back into Bronze/Silver/Gold, and `fact_sales` does not depend on it.

| Node | Relationship | Node |
|------|-------------|------|
| Customer | PLACED → | Order |
| Order | CONTAINS → | Product |
| Product | SUPPLIED_BY → | Supplier |
| Customer | LIVES_IN → | Address |
| Order | RETURNED_AS → | Return |

### Free Tool: Neo4j AuraDB Free

| Step | Action |
|------|--------|
| 1 | Go to `neo4j.com/cloud/aura` |
| 2 | Create free account |
| 3 | Create a free AuraDB instance |
| 4 | Download the connection credentials (URI + password) |
| 5 | Connect from Databricks using the `neo4j` Python driver |

## Section 5 — Cypher Query Language Basics

Cypher is Neo4j's query language. It is designed to look like an ASCII drawing of the graph.

```
SQL:     SELECT * FROM customers WHERE city = 'Mumbai'
Cypher:  MATCH (c:Customer {city: 'Mumbai'}) RETURN c
```

### Cypher Syntax — 4 key patterns

```cypher
-- 1. Find all customers
MATCH (c:Customer)
RETURN c.name, c.city
LIMIT 10

-- 2. Find all products a customer bought
MATCH (c:Customer {name: 'Raj'})-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN p.name, p.price

-- 3. Find customers who returned something
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:RETURNED_AS]->(r:Return)
RETURN c.name, r.reason

-- 4. Count orders per customer (like GROUP BY)
MATCH (c:Customer)-[:PLACED]->(o:Order)
RETURN c.name, COUNT(o) AS total_orders
ORDER BY total_orders DESC
```

### Reading from Neo4j into Databricks

```python
# Install: pip install neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("MATCH (c:Customer)-[:PLACED]->(o:Order) RETURN c.id, COUNT(o)")
    rows = [dict(r) for r in result]

# Convert to Spark DataFrame — land in the SAME sandbox/ location as the API demo,
# never bronze/. This is exploration output, not a pipeline table.
graph_df = spark.createDataFrame(rows)
graph_df.write.format("delta").mode("overwrite").save(sandbox_path + "/graph_customer_orders")
```

> **We will set up Neo4j AuraDB and run this during the hands-on (11:00 AM).**  
> For now — understand the concept. Cypher is easy once you think of it as drawing arrows.

## Recap

| Topic | Key Takeaway |
|-------|--------------|
| REST API | HTTP GET → JSON response → flatten → save to a landing table |
| Status codes | Always check 200 before processing |
| Rate limits | Don't hammer the API — add sleep, cache results |
| Graph DB | Nodes + Relationships — great for traversal queries |
| Cypher | `MATCH (a)-[:REL]->(b) RETURN a, b` — reads like a drawing |
| Neo4j AuraDB Free | Free cloud Neo4j — use for bootcamp demos |
| Scope | Side-exploration only — none of this feeds Bronze/Silver/Gold or `fact_sales` |

---

## Hands-on Next (11:00 AM – 1:00 PM, 2 hrs)
**Day3_HOL1_Public_API_GraphDB**

1. Call frankfurter.app → land exchange rates in a sandbox Delta table
2. Extend the API demo: multiple base currencies, historical rates, fill-in-the-blank exercises
3. Set up Neo4j AuraDB Free account
4. Load sample customer-product relationship data into Neo4j
5. Run Cypher queries in Neo4j browser
6. Read graph data from Neo4j into Databricks → save to a sandbox Delta table

---

## Next ILT — Day 3 ILT 2 (2:00 PM – 3:00 PM)
**Autoloader & Schema Evolution Concepts**